# OCR Evaluation — единый Kaggle-пайплайн (T4 x2)

Сравнение трёх OCR-моделей на OmniDocBench (страницы `academic_literature`):

1. **LightOnOCR-2-1B** — `lightonai/LightOnOCR-2-1B`, 1B, через `trust_remote_code`.
2. **DeepSeek-OCR** — `deepseek-ai/DeepSeek-OCR`, ~3B, grounding-промпт.
3. **olmOCR-7B** — `allenai/olmOCR-7B-0225-preview`, Qwen2-VL-7B fine-tuned.

Окружение: Kaggle Notebook, **GPU T4 x2** (2× sm_75 / 16 ГБ).

> Перед запуском в Kaggle:
> * Settings → Accelerator → **GPU T4 x2**.
> * Add-ons → Secrets → New Secret `HF_TOKEN` (read-токен HuggingFace).

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, pathlib, sys

REPO_URL  = "https://github.com/AStrateg2509/ocr_eval.git"
REPO_NAME = "ocr_eval"

if not pathlib.Path(REPO_NAME).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
else:
    subprocess.run(["git", "-C", REPO_NAME, "pull", "--ff-only"], check=False)

os.chdir(REPO_NAME)
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("CWD =", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y poppler-utils 2>&1 | tail -1

In [ ]:
# === flash-attn install (T4 = sm_75 поддерживает FA2) =====================
# Wheel выбираем по версии Python в текущем kernel'е и фиксированной версии torch (2.6.0).
# Если для конкретной (cpXY × torch × cu) пары wheel'а нет — install упадёт,
# и ниже модели поднимутся с attn_implementation='sdpa'. Это безопасный fallback.

import sys, subprocess, importlib

PY_TAG    = f"cp{sys.version_info.major}{sys.version_info.minor}"
TORCH_TAG = "torch2.6"
CUDA_TAG  = "cu12"
ABI       = "cxx11abiFALSE"

WHEEL = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.1.post4/"
    f"flash_attn-2.7.1.post4+{CUDA_TAG}{TORCH_TAG}{ABI}-{PY_TAG}-{PY_TAG}-linux_x86_64.whl"
)
print("flash-attn wheel:", WHEEL)

rc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall",
     WHEEL, "--no-build-isolation", "-q"],
    capture_output=True, text=True,
)
FLASH_ATTN_OK = (rc.returncode == 0)
if FLASH_ATTN_OK:
    try:
        import flash_attn
        print("flash-attn OK, version =", flash_attn.__version__)
    except Exception as e:
        print("wheel installed но импорт упал:", e)
        FLASH_ATTN_OK = False
else:
    print("flash-attn install FAILED — будем работать на sdpa.")
    print("stderr tail:", rc.stderr[-500:])

ATTN_IMPL = "flash_attention_2" if FLASH_ATTN_OK else "sdpa"
print("ATTN_IMPL =", ATTN_IMPL)

In [ ]:
# === GPU sanity ===========================================================
import torch
from src.utils import gpu_info, cuda_capabilities, supports_flash_attention

assert torch.cuda.is_available(), "GPU runtime не выбран!"
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("GPUs    :", gpu_info())

caps = cuda_capabilities()
if not all(cc[0] >= 7 for cc in caps):
    print("ВНИМАНИЕ: видна Pascal-карта (P100/Pascal). Рекомендован T4 x2 — "
          "иначе flash-attn придётся снимать.")
if not supports_flash_attention():
    print("ВНИМАНИЕ: GPU не поддерживает FA2; принудительно ATTN_IMPL='sdpa'")
    ATTN_IMPL = "sdpa"

# Sanity matmul — раннее обнаружение sm-несовместимости.
_ = (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).cpu()
print("matmul OK")

In [ ]:
# === HuggingFace auth =====================================================
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token from Kaggle Secrets — OK")
except Exception as e:
    print(f"Kaggle Secrets unavailable ({e!r}); fallback на env")
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("logged in to HF")
else:
    print("ВНИМАНИЕ: HF_TOKEN не найден — gated-модели/датасеты не загрузятся")

## 2. Датасет: chunked-загрузка OmniDocBench

OmniDocBench содержит ~1653 страницы. HuggingFace Hub лимитирует ~1000 запросов в минуту с IP,
поэтому делим на два этапа:

1. **JSON-разметка** одним вызовом.
2. **Изображения** — пачками по 200, с паузами 5 сек; после каждой пачки file-based verify.

In [ ]:
from src.dataset_loader import (
    download_omnidocbench_json,
    list_repo_images, list_image_filenames,
    download_omnidocbench_images_chunked,
    verify_downloaded_images,
)

DATA_ROOT = "data/OmniDocBench"
download_omnidocbench_json(DATA_ROOT, token=HF_TOKEN)
print("JSON скачан:", list(pathlib.Path(DATA_ROOT).glob("OmniDocBench*.json")))

In [ ]:
def progress(stage, **kw):
    print(f"[{stage}]", kw)

summary = download_omnidocbench_images_chunked(
    target_dir=DATA_ROOT,
    chunk_size=200,
    sleep_between_chunks=5.0,
    max_retries=5,
    token=HF_TOKEN,
    on_progress=progress,
)
summary

In [ ]:
expected = list_repo_images(token=HF_TOKEN)
report = verify_downloaded_images(DATA_ROOT, expected=expected)
print(f"present : {len(report['present'])}")
print(f"missing : {len(report['missing'])}")
print(f"broken  : {len(report['broken'])}")
if report['missing'][:5]:
    print("первые missing:", report['missing'][:5])
if report['broken'][:5]:
    print("первые broken :", report['broken'][:5])

if report['missing']:
    print("Докачиваем missing...")
    download_omnidocbench_images_chunked(DATA_ROOT, chunk_size=200,
                                         sleep_between_chunks=5.0, max_retries=5,
                                         token=HF_TOKEN)

## 3. Подвыборка: 100 страниц `academic_literature`, English

Один общий subset для всех трёх моделей — это даёт сопоставимые результаты.

In [ ]:
import json, pathlib
from src.dataset_loader import load_omnidocbench

items = load_omnidocbench(
    root=DATA_ROOT,
    page_types=["academic_literature"],
    languages=["english"],
    subset_size=100,
    seed=42,
    require_image_present=True,
)
print(f"отобрано: {len(items)} страниц")

subset_path = pathlib.Path("data/subset.json")
subset_path.parent.mkdir(parents=True, exist_ok=True)
subset_path.write_text(
    json.dumps([x.to_dict() for x in items], ensure_ascii=False),
    encoding="utf-8",
)
print("сохранено:", subset_path)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

first = items[0]
img = Image.open(Path(DATA_ROOT) / first.image_path).convert("RGB")
fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(img); ax.axis("off")
ax.set_title(f"{first.page_id}  ·  {first.page_type}  ·  {first.language}")
plt.show()

## 4. Общие хелперы

In [ ]:
from src.utils import (
    load_config, JsonlWriter, Timer, cuda_free, gpu_info,
    already_processed_ids, drop_error_records, read_jsonl,
)
from src.io_records import PredictionRecord
from src.dataset_loader import GroundTruth

import torch, traceback, json
from pathlib import Path
from PIL import Image

DATA_ROOT = Path("data/OmniDocBench")
subset = [GroundTruth(**rec) for rec in json.loads(
    Path("data/subset.json").read_text(encoding="utf-8"))]
print("subset:", len(subset), "страниц")

## 5. LightOnOCR-2-1B

Загружаем через `trust_remote_code=True` с try-chain Auto-классов:
**Vision2Seq → ImageTextToText → CausalLM → Mistral3ForConditionalGeneration**.

Логика: первые 3 респектят `auto_map` из репо LightOn (если он там настроен) —
тогда поднимется кастомный класс LightOn'а, ожидающий `vision_encoder` (а не `vision_tower`),
и LOAD-REPORT mismatch не возникнет.

In [ ]:
cfg = load_config("configs/lightonocr.yaml")
MODEL_REPO = cfg["model"]["hf_repo"]
DTYPE = getattr(torch, cfg["model"]["torch_dtype"])
EFFECTIVE_ATTN = ATTN_IMPL  # из ячейки flash-attn

from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(
    MODEL_REPO,
    trust_remote_code=cfg["model"]["trust_remote_code"],
    token=HF_TOKEN,
)
print("processor:", type(processor).__name__)
print("attn impl :", EFFECTIVE_ATTN)

In [ ]:
# --- LightOnOCR loader: try-chain ---
from transformers import (
    AutoModelForVision2Seq,
    AutoModelForCausalLM,
    AutoModel,
)
try:
    from transformers import AutoModelForImageTextToText
except Exception:
    AutoModelForImageTextToText = None
try:
    from transformers import Mistral3ForConditionalGeneration
except Exception:
    Mistral3ForConditionalGeneration = None

common_kw = dict(
    trust_remote_code=cfg["model"]["trust_remote_code"],
    torch_dtype=DTYPE,
    device_map=cfg["model"]["device_map"],
    token=HF_TOKEN,
    attn_implementation=EFFECTIVE_ATTN,
)

candidates = [
    ("AutoModelForVision2Seq",     AutoModelForVision2Seq),
    ("AutoModelForImageTextToText",AutoModelForImageTextToText),
    ("AutoModelForCausalLM",       AutoModelForCausalLM),
    ("Mistral3ForConditionalGeneration", Mistral3ForConditionalGeneration),
]

model = None
last_err = None
for label, cls in candidates:
    if cls is None:
        print(f"  [skip] {label}: класса нет в этой transformers")
        continue
    try:
        print(f"  [try]  {label}")
        model = cls.from_pretrained(MODEL_REPO, **common_kw).eval()
        if not hasattr(model, "generate"):
            print(f"  [bad]  {label}: нет .generate(), пробую следующий")
            del model; model = None
            continue
        print(f"  [OK]   loaded with {label} → {type(model).__name__}")
        break
    except Exception as e:
        last_err = e
        print(f"  [fail] {label}: {type(e).__name__}: {str(e)[:200]}")

if model is None:
    raise RuntimeError(f"не удалось загрузить LightOnOCR; last_err={last_err}")
print(gpu_info())

In [ ]:
out_path = Path(cfg["output"]["results_dir"]) / "predictions.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)

# Сбрасываем ошибочные записи прошлых запусков, чтобы они переобработались.
print("drop_error_records:", drop_error_records(out_path))
done = already_processed_ids(out_path)
print(f"уже обработано: {len(done)} / {len(subset)}")

PROMPT  = cfg["inference"]["prompt"]
MAX_NEW = cfg["inference"]["max_new_tokens"]

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model="lightonocr")
        try:
            img = Image.open(img_path).convert("RGB")
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": PROMPT},
                ],
            }]
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt")
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            with Timer("infer") as t, torch.no_grad():
                gen = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW,
                    do_sample=cfg["inference"]["do_sample"],
                    num_beams=cfg["inference"]["num_beams"],
                )
            input_len = inputs["input_ids"].shape[1]
            out = processor.batch_decode(gen[:, input_len:], skip_special_tokens=True)[0]
            rec.full_text = out
            rec.raw_output = out
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        w.write(rec.to_dict())

print("→", out_path, "записей:", sum(1 for _ in open(out_path)))
del model, processor; cuda_free(); print(gpu_info())

## 6. DeepSeek-OCR

Главный фикс из прошлой итерации: `model.infer(...)` НЕ возвращает markdown как строку —
он пишет файл `<output_path>/result.mmd`. Поэтому ставим `save_results=True` и читаем
результат с диска.

На T4 x2 `device_map=auto` сама раскладывает 3B-модель.

In [ ]:
from transformers import AutoModel, AutoTokenizer

cfg = load_config("configs/deepseek_ocr.yaml")
MODEL_REPO = cfg["model"]["hf_repo"]
DTYPE = getattr(torch, cfg["model"]["torch_dtype"])

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    device_map=cfg["model"]["device_map"],
    token=HF_TOKEN,
).eval()
print(gpu_info())

In [ ]:
out_dir = Path(cfg["output"]["results_dir"])
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "predictions.jsonl"

print("drop_error_records:", drop_error_records(out_path))
done = already_processed_ids(out_path)
print(f"уже обработано: {len(done)} / {len(subset)}")

RESULT_FILENAME = cfg["inference"].get("result_filename", "result.mmd")

def _read_deepseek_result(folder: Path) -> str:
    candidates = [folder / RESULT_FILENAME,
                  folder / "result.mmd",
                  folder / "result.txt",
                  folder / "output.mmd"]
    for c in candidates:
        if c.exists() and c.stat().st_size > 0:
            return c.read_text(encoding="utf-8", errors="ignore")
    for f in folder.glob("*.mmd"):
        return f.read_text(encoding="utf-8", errors="ignore")
    return ""

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model="deepseek_ocr")
        try:
            tmp_out = out_dir / gt.page_id
            tmp_out.mkdir(parents=True, exist_ok=True)
            with Timer("infer") as t:
                _ret = model.infer(
                    tokenizer,
                    prompt=cfg["inference"]["prompt"],
                    image_file=str(img_path),
                    output_path=str(tmp_out),
                    base_size=cfg["inference"]["base_size"],
                    image_size=cfg["inference"]["image_size"],
                    crop_mode=cfg["inference"]["crop_mode"],
                    save_results=cfg["inference"]["save_results"],
                    test_compress=cfg["inference"]["test_compress"],
                )
            text = _read_deepseek_result(tmp_out)
            if not text and isinstance(_ret, str):
                text = _ret
            rec.full_text = text
            rec.raw_output = text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        w.write(rec.to_dict())

print("→", out_path, "записей:", sum(1 for _ in open(out_path)))
del model, tokenizer; cuda_free(); print(gpu_info())

## 7. olmOCR (Qwen2-VL-7B fine-tuned)

T4 x2: `float16` + `flash_attention_2` (если поставился) + `device_map='auto'` —
7B расщепляется между двумя картами. `max_image_long_side=1536`.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

cfg = load_config("configs/olmocr.yaml")
MODEL_REPO = cfg["model"]["hf_repo"]
DTYPE = getattr(torch, cfg["model"]["torch_dtype"])

# attn_impl из конфига, но если flash-attn не встал — фолбек на sdpa
_cfg_attn = cfg["model"]["attn_implementation"]
EFFECTIVE_ATTN_OLM = _cfg_attn if (_cfg_attn != "flash_attention_2" or FLASH_ATTN_OK) else "sdpa"
print("olmOCR attn impl:", EFFECTIVE_ATTN_OLM)

processor = AutoProcessor.from_pretrained(MODEL_REPO, token=HF_TOKEN)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_REPO,
    torch_dtype=DTYPE,
    device_map=cfg["model"]["device_map"],
    attn_implementation=EFFECTIVE_ATTN_OLM,
    low_cpu_mem_usage=cfg["model"]["low_cpu_mem_usage"],
    token=HF_TOKEN,
).eval()
print(gpu_info())

In [ ]:
out_path = Path(cfg["output"]["results_dir"]) / "predictions.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)

print("drop_error_records:", drop_error_records(out_path))
done = already_processed_ids(out_path)
print(f"уже обработано: {len(done)} / {len(subset)}")

MAX_NEW   = cfg["inference"]["max_new_tokens"]
TEMP      = cfg["inference"]["temperature"]
LONG_SIDE = cfg["inference"]["max_image_long_side"]

OLMOCR_PROMPT = (
    "Below is the image of one page of a document. Just return the plain text "
    "representation of this document as if you were reading it naturally. "
    "Convert equations to LaTeX and tables to HTML. Do not hallucinate."
)

def _resize(img, long_side):
    w, h = img.size
    if max(w, h) <= long_side:
        return img
    if w >= h:
        new = (long_side, int(h * long_side / w))
    else:
        new = (int(w * long_side / h), long_side)
    return img.resize(new, Image.LANCZOS)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model="olmocr")
        try:
            img = _resize(Image.open(img_path).convert("RGB"), LONG_SIDE)
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": OLMOCR_PROMPT},
                ],
            }]
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt").to(model.device)
            with Timer("infer") as t, torch.no_grad():
                gen = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW,
                    do_sample=TEMP > 0,
                    temperature=TEMP if TEMP > 0 else 1.0,
                )
            out = processor.batch_decode(
                gen[:, inputs.input_ids.shape[1]:],
                skip_special_tokens=True,
            )[0]
            rec.full_text = out
            rec.raw_output = out
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        w.write(rec.to_dict())

print("→", out_path, "записей:", sum(1 for _ in open(out_path)))
del model, processor; cuda_free(); print(gpu_info())

## 8. Быстрая верификация выходов

Не метрики (это следующий шаг), а контроль того, что инференс реально записал данные.

In [ ]:
import pandas as pd

rows = []
for model_name in ("lightonocr", "deepseek_ocr", "olmocr"):
    p = Path("results") / model_name / "predictions.jsonl"
    if not p.exists():
        rows.append({"model": model_name, "records": 0, "errors": 0,
                     "avg_time_s": 0.0, "avg_text_len": 0})
        continue
    recs = read_jsonl(p)
    errors = sum(1 for r in recs if r.get("error"))
    times = [r["inference_time_s"] for r in recs if not r.get("error")]
    lens  = [len(r.get("full_text", "")) for r in recs if not r.get("error")]
    rows.append({
        "model": model_name,
        "records": len(recs),
        "errors": errors,
        "avg_time_s": round(sum(times) / max(1, len(times)), 2),
        "avg_text_len": round(sum(lens) / max(1, len(lens)), 0),
    })

pd.DataFrame(rows)

---
**Готово.** Дальше — модули метрик (Text / Table / IoU / Recall@5 / Overall)
и сводный CSV `results/summary.csv`.